In [ ]:
# Paste this complete file into the first code cell of a fresh Colab session.
# Select Runtime > Change runtime type > T4 GPU before running.
from google.colab import drive
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # Replace this.
REPO_DIR = Path("/content/retinocheck-ai")
DRIVE_ROOT = Path("/content/drive/MyDrive/RetinoCheckAI")

if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repository URL first.")
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("These pins target Python 3.11–3.13. Select a matching Colab runtime.")

drive.mount("/content/drive")
for folder in ("checkpoints", "reports"):
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)

# Reuse the clone when rerunning; avoid overwriting local changes with a pull.
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / ".git").exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git clone. Choose another REPO_DIR.")
else:
    remote = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip()
    if remote != REPO_URL:
        raise RuntimeError(f"Existing clone has a different origin: {remote}")
    print("Using existing clone. Commit:", subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# CUDA 12.4 wheels: the official matching PyTorch 2.6 / torchvision 0.21 pair.
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch==2.6.0", "torchvision==0.21.0",
                "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

# A separate Python process checks the newly installed packages, even if the
# notebook kernel already imported older versions. Allocate a GPU tensor too.
subprocess.run([sys.executable, "-c", """
import torch, torchvision, timm, datasets, sklearn, gradio, pytorch_grad_cam
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU. Select a T4 GPU runtime and rerun setup.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU tensor check:', (torch.ones(1, device='cuda') + 1).item())
"""], check=True)
print("Setup complete. If packages were already imported, restart the session before training.")


In [ ]:
# All analysis and splitting logic is in src/data.py.
# Dataset annotations and model outputs are for research only, never medical advice.
from src.config import Config
from src.data import (
    load_source, describe_dataset, plot_grade_counts, show_grade_samples,
    get_or_create_splits, verify_split_proportions,
)

config = Config()
print("Frozen split file:", config.split_file)
# For a final experiment, set config.revision to an immutable HF commit.
# If HF fails: obtain Kaggle APTOS 2019 train.csv and train_images/,
# set config.local_data to their parent directory, and use a distinct split_file.
source, labels, identity = load_source(config)
print("Dataset fingerprint:", identity)


In [ ]:
# Prints columns, total images, grade counts and proportions.
# Scans EVERY original image for min/median/mean/max/std width and height.
# Images are decoded one at a time; the full image set is not held in RAM.
sizes = describe_dataset(source, labels)


In [ ]:
# Class imbalance: annotations 0 through 4.
grade_figure = plot_grade_counts(labels)


In [ ]:
# Three original images per grade, sampled reproducibly without replacement.
# The HF mirror may already be resized; original source detail cannot be restored.
sample_figure = show_grade_samples(source, labels, seed=config.seed, samples_per_grade=3)


In [ ]:
# First run: create stratified 70/15/15 splits and write indices to Drive.
# Later runs: read the SAME file; never overwrite or silently regenerate it.
# Both model backbones use this shared path through make_loaders().
# A changed source fingerprint, seed, ratios or invalid indices raises an error.
manifest = get_or_create_splits(config, labels, identity)


In [ ]:
# Verify full coverage, no overlap, all grades present and preserved proportions.
# Counts may differ slightly from exact 70/15/15 due to integer rounding.
split_report = verify_split_proportions(labels, manifest)


In [ ]:
# Verify a rerun reads identical indices, including test membership.
reloaded_manifest = get_or_create_splits(config, labels, identity)
assert reloaded_manifest["indices"] == manifest["indices"]
print("PASS: saved train/val/test indices reload unchanged.")
print("Keep a backup of:", config.split_file)
# This is image-level stratification: patient independence and absence of
# duplicate images are NOT established. Do not tune models on the test set.
